In [1]:
import polars as pl
import ast
import numpy as np
import time
import os
from tqdm import tqdm
from multiprocessing import Pool
import itertools
import matplotlib.pyplot as plt
import gc
from collections import Counter
import math
import keras
from scipy import sparse
from keras import models, layers
import keras

In [ ]:
class CustomDataGenerator(keras.utils.Sequence):
    def __init__(self,
                src_data,
                batch_size, 
                dataset_type,
                shuffle=True,
                **kwargs):
        super().__init__(**kwargs)
        self.src_data = src_data
        self.batch_size = batch_size
        assert dataset_type in ["train", "validation", "test"]
        self.dataset_type = dataset_type
        self.shuffle = shuffle
        self.dataset = self.read_dataset()
    
    def read_dataset(self):
        dataset = pl.read_parquet(os.path.join(self.src_data, 
                                               f"data_{self.dataset_type}.parquet"))
        dataset = dataset.filter(pl.col("len_player1_card").is_in(range(1, 5)))
        print(f"len_player1_card values in {self.dataset_type}:", dataset["len_player1_card"].value_counts())
        print(f"The shape of the dataset {self.dataset_type} is ({dataset.shape[0]:,}; {dataset.shape[1]})")
        return dataset.sample(fraction=1.0, shuffle=True)

    def on_epoch_end(self):
        # print("Shuffling dataset on the end of epoch")
        # start_time_shuffle = time.time()
        if self.shuffle:
            self.dataset = self.dataset.sample(fraction=1.0, shuffle=True)
            gc.collect()
        # print(f"Shuffling took {time.time() - start_time_shuffle} seconds")

    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)

    # def normalize_dataset(self, dataset: pl.DataFrame):
    #     bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
    #     dataset = dataset.with_columns(
    #         [
    #             pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Int8
    #             (pl.col("player1_action_max_card") + 1) / 52,  # Normalize player1_action_max_card
    #             (pl.col("action_max_card") + 1) / 52,  # Normalize action_max_card
    #             (pl.col("len_player1_card") - 1) / 12,  # Normalize len_player1_card
    #         ]
    #     )
    #     return dataset
    
    def normalize_dataset(self, dataset: pl.DataFrame):
        bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
        dataset = dataset.with_columns(
            [
                pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Int8
                (pl.col("player1_action_max_card") + 1) / 52,  # Normalize player1_action_max_card
                (pl.col("action_max_card") + 1) / 52,  # Normalize action_max_card
                (pl.col("len_player1_card") - 1) / 12,  # Normalize len_player1_card
            ] + 
            [pl.col(f"player2_card_info_{i}") / 4 for i in range(13)] +  # Normalize rank counts by dividing by 4
            [pl.col(f"player2_card_info_{i}") / 13 for i in range(13, 17)] +  # Normalize suit counts by dividing by 13
            [pl.col(f"deck_remain_info_{i}") / 4 for i in range(13)] +  # Normalize rank counts by dividing by 4
            [pl.col(f"deck_remain_info_{i}") / 13 for i in range(13, 17)] +  # Normalize suit counts by dividing by 13
            [pl.col(f"player2_n_sequence_double_sequence_info_{i}") / 42 for i in range(10)] +  # Normalize first 10 sequence info by dividing by 42
            [pl.col(f"player2_n_sequence_double_sequence_info_{i}") / 6 for i in range(10, 12)] +  # Normalize last 2 sequence info by dividing by 6
            [pl.col(f"deck_remain_n_sequence_double_sequence_info_{i}") / 31104 for i in range(10)] +  # Normalize first 10 sequence info by dividing by 31104
            [pl.col(f"deck_remain_n_sequence_double_sequence_info_{i}") / 2592 for i in range(10, 12)]  # Normalize last 2 sequence info by dividing by 2592
        )
        return dataset

    def __getitem__(self, index):
        # batch_data = sparse.load_npz(os.path.join(self.src_data, 
        #                     f"data_{self.dataset_type}_batch_{index}.npz")).toarray()
        batch_data = self.dataset[(index * self.batch_size): ((index + 1) * self.batch_size)]
        batch_data = self.normalize_dataset(batch_data)
        batch_data = batch_data.to_numpy()
        X = batch_data[:, :-1]
        y = batch_data[:, -1]
        return X, y


src_data = r"D:\Project\TLMN\preprocess"
batch_size = 4096
data_train_generator = CustomDataGenerator(
    src_data=src_data,
    batch_size=batch_size,
    dataset_type="train")
data_validation_generator = CustomDataGenerator(
    src_data=src_data,
    batch_size=batch_size,
    dataset_type="validation")

In [2]:
# class CustomDataGenerator(keras.utils.Sequence):
#     def __init__(self,
#                  src_data, 
#                  dataset_type,
#                  **kwargs):
#         super().__init__(**kwargs)
#         self.src_data = src_data
#         assert dataset_type in ["train", "validation", "test"]
#         self.dataset_type = dataset_type
    
#     def on_epoch_end(self):
#         pass

#     def __len__(self):
#         return len(os.listdir(self.src_data))

#     def __getitem__(self, index):
#         batch_data = sparse.load_npz(os.path.join(self.src_data, 
#                             f"data_{self.dataset_type}_batch_{index}.npz")).toarray()
#         X = batch_data[:, :-1]
#         y = batch_data[:, -1]
#         return X, y

# data_train_generator = CustomDataGenerator(
#     src_data=r"/kaggle/input/big-data-train/data_train", 
#     dataset_type="train")
# data_validation_generator = CustomDataGenerator(
#     src_data=r"/kaggle/input/big-data-validation/data_validation", 
#     dataset_type="validation")

In [ ]:
model = models.Sequential()
model.add(layers.Input(shape=(data_train_generator[0][0].shape[1],)))
# model.add(layers.Dense(2048, kernel_initializer='he_normal'))
# model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(1024, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(512, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(256, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(128, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(64, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(32, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(16, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
# model.add(layers.Dense(1, activation="linear"))
model.add(layers.Dense(1, activation="sigmoid"))

model.summary()

In [ ]:
# Define a residual block for dense layers
def dense_residual_block(x, units, use_bias=True):
    shortcut = x
    
    # First dense layer
    x = layers.Dense(units, kernel_initializer='he_normal', use_bias=use_bias)(x)
    x = layers.BatchNormalization()(x)
    x = layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25))(x)
    
    # Second dense layer
    x = layers.Dense(units, kernel_initializer='he_normal', use_bias=use_bias)(x)
    x = layers.BatchNormalization()(x)
    
    # Project shortcut if needed
    if shortcut.shape[-1] != units:
        shortcut = layers.Dense(units, kernel_initializer='he_normal', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add skip connection
    x = layers.Add()([shortcut, x])
    x = layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25))(x)
    
    return x

# Build ResNet-style model
inputs = layers.Input(shape=(data_train_generator[0][0].shape[1],))
x = inputs

# Initial projection
x = layers.Dense(2048, kernel_initializer='he_normal')(x)
x = layers.BatchNormalization()(x)
x = layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25))(x)

# Residual blocks with decreasing units
x = dense_residual_block(x, 2048)
x = dense_residual_block(x, 1024)
x = dense_residual_block(x, 512)
x = dense_residual_block(x, 256)
x = dense_residual_block(x, 128)
x = dense_residual_block(x, 64)
x = dense_residual_block(x, 32)

# Final layers
x = layers.Dense(16, kernel_initializer='he_normal')(x)
x = layers.BatchNormalization()(x)
x = layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25))(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

In [ ]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=r"D:\Project\TLMN\MLP_TLMN.keras",
    monitor='val_mae',
    mode='min',
    save_best_only=True)
# model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
# model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
#     filepath=r"/kaggle/working/TLMN_see_opponent_card_20k_games.keras",
#     monitor='val_accuracy',
#     mode='max',
#     save_best_only=True)

history = model.fit(data_train_generator, 
                    validation_data=data_validation_generator,
                    epochs=50)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Mean Squared Error')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#Evaluate the model on the testing set
from keras.models import load_model

best_model = load_model(r"F:\Epoch_3_TLMN_see_opponent_card_20k_games_NVIDIA_Quadro_P2200.keras")
data_test_generator = CustomDataGenerator(
    src_data=r"F:\big_data\data_test", 
    dataset_type="test")
best_model.evaluate(data_test_generator)